In [ ]:
from google.colab import drive
drive.mount("/gdrive")
current_dir = "/gdrive/My\\ Drive/Challenge2_nuova"
%cd $current_dir

Drive already mounted at /gdrive; to attempt to forcibly remount, call drive.mount("/gdrive", force_remount=True).
/gdrive/.shortcut-targets-by-id/1cd3Xbh9ZKbBedPV57qHiIKUqq2PyuVP_/Challenge2_nuova


In [ ]:
# Set seed for reproducibility
SEED = 42

# Import necessary libraries
import os

# Set environment variables before importing modules
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
import numpy as np

# Set seeds for random number generators in NumPy and Python
np.random.seed(SEED)
random.seed(SEED)

# Import PyTorch
import torch
torch.manual_seed(SEED)
from torch import nn
from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision.transforms import v2 as transforms
from torch.utils.data import TensorDataset, DataLoader
!pip install torchview
from torchview import draw_graph
from tqdm.notebook import tqdm

# Configurazione di TensorBoard e directory
logs_dir = "tensorboard"
!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p models

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Import other libraries
import cv2
import copy
import shutil
from itertools import product
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from PIL import Image
import matplotlib.gridspec as gridspec
import requests
from io import BytesIO


# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline

PyTorch version: 2.9.0+cu126
Device: cuda


In [ ]:
# @title **Model Definition**

import torch
import torch.nn as nn
import torchvision
from torchsummary import summary
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_score, recall_score

!pip install transformers

import torch
import torch.nn as nn
from torchvision import transforms
from transformers import AutoModel

# Configurazione Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# --- MODEL DEFINITION (Phikon ViT) ---
class PhikonModel(nn.Module):
    """Phikon adapted for transfer learning."""

    def __init__(self, num_classes, dropout_rate=0.3, freeze_backbone=True):
        super().__init__()

        # Load Phikon backbone (which is a ViT-based model)
        self.backbone = AutoModel.from_pretrained("owkin/phikon")

        # Freeze backbone parameters if specified
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # Get the input features size for the classification head from the model's configuration
        # For ViT models, this is typically the hidden_size of the transformer output.
        in_features = self.backbone.config.hidden_size

        # Define a new classification head for the PhikonModel
        # This head will replace the original classification mechanism of the pretrained model
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        # Pass input through the backbone. AutoModel.from_pretrained("owkin/phikon")
        # returns a BaseModelOutputWithPooling object for ViT models.
        outputs = self.backbone(x)

        # Extract the pooled output (typically the [CLS] token's embedding after pooling),
        # which is suitable for classification. If pooler_output is None or not available,
        # fall back to the first token of the last_hidden_state.
        pooled_output = outputs.pooler_output if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None else outputs.last_hidden_state[:, 0]

        # Pass the pooled output through the custom classification head
        logits = self.classifier(pooled_output)
        return logits

    def unfreeze_last_stage(self):
        """Unfreeze the last two transformer encoder layers for fine-tuning."""
        # The ViT encoder layers are typically within `self.backbone.encoder.layer`
        # for AutoModel.from_pretrained(ViT).
        # We unfreeze the last two layers as a 'last stage' equivalent for fine-tuning.
        for param in self.backbone.encoder.layer[-2:].parameters():
            param.requires_grad = True
        print("Info: Last two transformer encoder layers unfrozen for training.")

Device: cuda


In [ ]:
# @title **Part 1: Data Preparation & Patching**
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import shutil

# --- CONFIGURATION ---
SOURCE_DATA_DIR = "challenge_data/train_data"
LABELS_CSV_PATH = "challenge_data/train_labels.csv"
TARGET_DATA_DIR = "dataset_patched"

# Patching Parameters
PATCH_SIZE = 224
STRIDE = 224
# 1% Threshold (Permissive to avoid losing Triple Negative)
TISSUE_THRESHOLD_RATIO = 0.01

# --- BLACKLIST (Corrupted images to ignore) ---
IMGS_TO_REMOVE = [1, 5, 8, 12, 18, 20, 22, 25, 27, 28, 36, 47, 48, 52,
                        62, 78, 85, 90, 94, 95, 126, 129, 130, 133, 136, 138,
                        148, 150, 155, 159, 161, 175, 178, 179, 180, 187, 189, 193, 196, 222, 237,
                        251, 253, 254, 263, 268, 270, 286, 293, 313, 319, 333, 342, 344, 346, 355,
                        368, 371, 376, 380, 390, 393, 407, 410, 415, 424, 440, 443, 453, 459, 463,
                        486, 497, 498, 499, 509, 521, 530, 531, 533, 537, 540, 544, 547, 557, 558,
                        560, 565, 567, 572, 576, 578, 580, 586, 602, 603, 607, 609, 614, 620, 623,
                        629, 635, 639, 643, 644, 645, 646, 656, 657, 658, 670, 673, 675]

def create_patches_pipeline(source_dir, target_dir, csv_path):
    source_path = Path(source_dir)
    target_path = Path(target_dir)

    # 1. Load Labels
    print("Loading labels...")
    try:
        df = pd.read_csv(csv_path)
        labels_map = pd.Series(df.iloc[:, 1].values, index=df.iloc[:, 0].values).to_dict()
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return

    # 2. Reset Output Directory
    if target_path.exists():
        shutil.rmtree(target_path)
    target_path.mkdir(parents=True, exist_ok=True)

    # 3. Get Files
    all_files = list(source_path.glob("*.*"))
    img_files = [f for f in all_files if f.name.startswith("img_")]

    print(f"Total images on disk: {len(img_files)}")

    count_saved = 0
    count_skipped = 0

    # 4. Processing Loop
    for img_path in tqdm(img_files):
        # --- A. BLACKLIST CHECK ---
        try:
            file_id = int(img_path.stem.replace("img_", ""))
            if file_id in IMGS_TO_REMOVE:
                count_skipped += 1
                continue
        except ValueError:
            pass

        # --- B. LABEL CHECK ---
        img_name = img_path.name
        label = labels_map.get(img_name) or labels_map.get(img_path.stem)
        if label is None:
            continue

        # --- C. FOLDER CREATION ---
        safe_label = str(label).replace(" ", "_").replace("(", "").replace(")", "").replace("+", "plus")
        class_dir = target_path / safe_label
        class_dir.mkdir(parents=True, exist_ok=True)

        # --- D. IMAGE LOAD ---
        image = cv2.imread(str(img_path))
        if image is None:
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # RGB 3 channels

        # --- E. MASK LOAD (Only for filtering) ---
        mask_path = source_path / f"mask_{img_path.stem.replace('img_', '')}.png"
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) if mask_path.exists() else None

        h, w, _ = image.shape
        if h < PATCH_SIZE or w < PATCH_SIZE:
            continue

        # --- F. TILING ---
        p_idx = 0
        for y in range(0, h - PATCH_SIZE + 1, STRIDE):
            for x in range(0, w - PATCH_SIZE + 1, STRIDE):
                patch = image[y:y+PATCH_SIZE, x:x+PATCH_SIZE]

                # Validity Logic (Mask OR White-heuristic)
                is_valid = False
                if mask is not None:
                    patch_mask = mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
                    if (np.sum(patch_mask > 0) / (PATCH_SIZE**2)) > TISSUE_THRESHOLD_RATIO:
                        is_valid = True
                else:
                    gray = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
                    if (np.sum(gray > 220) / (PATCH_SIZE**2)) < 0.99:
                        is_valid = True

                if is_valid:
                    cv2.imwrite(
                        str(class_dir / f"{img_path.stem}_p{p_idx}.png"),
                        cv2.cvtColor(patch, cv2.COLOR_RGB2BGR)
                    )
                    count_saved += 1
                    p_idx += 1

    print(f"\nDone! Saved {count_saved} patches. Skipped {count_skipped} dirty images.")

create_patches_pipeline(SOURCE_DATA_DIR, TARGET_DATA_DIR, LABELS_CSV_PATH)


Loading labels...
Total images on disk: 691


100%|██████████| 691/691 [00:54<00:00, 12.64it/s]


Done! Saved 2944 patches. Skipped 114 dirty images.


In [ ]:
# --- CONFIGURATION & INITIAL TRAINING (Warmup) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_shape = (3, PATCH_SIZE, PATCH_SIZE)

# Model Instance
num_classes = len(full_dataset.classes)
model = PhikonModel(
    num_classes=num_classes,
    dropout_rate=DROPOUT_RATE,
    freeze_backbone=True
).to(device)

# Optimizer & Scaler
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

In [ ]:
# @title **Part 2 & 3: Augmentation, Dataset, Model Setup & Training Loop**

import torch
import torch.nn as nn
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import ReduceLROnPlateau

# --- 1. DATA AUGMENTATION PIPELINE ---
# Histology-specific augmentation pipeline (Training)
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(180),
    transforms.RandomApply([transforms.ElasticTransform(alpha=50.0, sigma=5.0)], p=0.2),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 1.25))], p=0.2),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1)], p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Base pipeline (Validation)
val_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(180),
    transforms.RandomApply([transforms.ElasticTransform(alpha=50.0, sigma=5.0)], p=0.2),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 1.25))], p=0.2),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1)], p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 2. DATASET LOADING & SPLIT ---
data_dir = "dataset_patched"  # Make sure this folder exists
full_dataset = datasets.ImageFolder(root=data_dir)

print(f"Classes found: {full_dataset.classes}")

# Split 80/20
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
generator = torch.Generator().manual_seed(42)  # Reproducibility
train_subset, val_subset = random_split(full_dataset, [train_size, val_size], generator=generator)

# --- 3. HELPER DATASET CLASSES ---
class TransformSubset(torch.utils.data.Dataset):
    """Applies transformations to the training subset."""
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

class TransformSubsetWithOriginalID(torch.utils.data.Dataset):
    """Also returns the original image ID for aggregation during validation."""
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        original_img_pil, label_idx = self.subset[index]

        # Retrieve original path from the underlying dataset (ImageFolder)
        # random_split internal structure: subset.dataset -> ImageFolder
        # subset.indices[index] -> original index in ImageFolder
        original_index = self.subset.indices[index]
        img_path, _ = self.subset.dataset.samples[original_index]

        # Extract the ID (e.g., from 'dataset_patched/Luminal/img_100_p0.png' -> 'img_100')
        original_img_id = Path(img_path).stem.split('_p')[0]

        if self.transform:
            transformed_img = self.transform(original_img_pil)
        else:
            transformed_img = transforms.ToTensor()(original_img_pil)

        return transformed_img, label_idx, original_img_id

    def __len__(self):
        return len(self.subset)

# Create final datasets
train_data = TransformSubset(train_subset, transform=train_transforms)
val_data = TransformSubsetWithOriginalID(val_subset, transform=val_transforms)

# Create DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Training patches: {len(train_data)}")
print(f"Validation patches: {len(val_data)}")

# --- 4. TRAINING SETUP (Loss & Weights) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Compute class weights (Correct and moved here)
y_all_indices = full_dataset.targets
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_all_indices),
    y=y_all_indices
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print("\nClass Weights:", class_weights_tensor)

# Loss Function (Single definition!)
criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    label_smoothing=0.1
)

# Hyperparameters
LEARNING_RATE = 1e-4  # Lowered for stable fine-tuning
EPOCHS = 30
PATIENCE = 10
DROPOUT_RATE = 0.5

# --- 5. TRAINING FUNCTIONS ---
def train_one_epoch(model, train_loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    all_preds, all_targets = [], []

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(inputs)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * inputs.size(0)
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    epoch_f1 = f1_score(all_targets, all_preds, average='weighted')

    return epoch_loss, epoch_acc, epoch_f1

def validate_one_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0

    # Dictionaries for aggregation
    img_votes = {}  # {img_id: [vote1, vote2...]}
    img_labels = {}  # {img_id: ground_truth_label}

    with torch.no_grad():
        for inputs, targets, img_ids in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(inputs)
                loss = criterion(logits, targets)

            running_loss += loss.item() * inputs.size(0)
            preds = logits.argmax(dim=1).cpu().numpy()
            targets_np = targets.cpu().numpy()

            # Vote aggregation
            for i, img_id in enumerate(img_ids):
                if img_id not in img_votes:
                    img_votes[img_id] = []
                    img_labels[img_id] = targets_np[i]
                img_votes[img_id].append(preds[i])

    # Compute aggregated metrics (Slide-Level)
    final_preds, final_targets = [], []
    for img_id in img_votes:
        votes = img_votes[img_id]
        # Majority Voting
        winner = np.bincount(votes).argmax()
        final_preds.append(winner)
        final_targets.append(img_labels[img_id])

    slide_acc = accuracy_score(final_targets, final_preds)
    slide_f1 = f1_score(final_targets, final_preds, average='weighted')
    patch_loss = running_loss / len(val_loader.dataset)

    return patch_loss, slide_acc, slide_f1

def fit(model, train_loader, val_loader, epochs, criterion, optimizer, scaler, device,
        patience=0, experiment_name="exp"):

    writer = SummaryWriter(f"logs/{experiment_name}")
    best_f1 = 0.0
    patience_counter = 0

    # Initialize history dictionary
    history = {
        'train_loss': [],
        'train_acc': [],
        'train_f1': [],
        'val_loss': [],
        'val_acc': [],
        'val_f1': []
    }

    # --- SCHEDULER ---
    # If the Validation Loss doesn't drop for 5 epochs, then divide the LR by 2
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    print(f"\nStarting training for {epochs} epochs...")

    for epoch in range(1, epochs + 1):
        t_loss, t_acc, t_f1 = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
        v_loss, v_acc, v_f1 = validate_one_epoch(model, val_loader, criterion, device)

        # Store metrics in history
        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['train_f1'].append(t_f1)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)
        history['val_f1'].append(v_f1)

        # Log current Learning Rate
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:03d} | Train Loss: {t_loss:.4f} f1: {t_f1:.4f} | Val Loss: {v_loss:.4f} f1: {v_f1:.4f} (Slide-Level)")

        writer.add_scalar('Acc/Train', t_acc, epoch)
        writer.add_scalar('Acc/Val', v_acc, epoch)
        writer.add_scalar('Loss/Train', t_loss, epoch)
        writer.add_scalar('Loss/Val', v_loss, epoch)
        writer.add_scalar('F1/Train', t_f1, epoch)
        writer.add_scalar('F1/Val', v_f1, epoch)

        scheduler.step(v_loss)

        # Early Stopping on Validation Accuracy (Slide-Level)
        if v_f1 > best_f1:
            best_f1 = v_f1
            patience_counter = 0
            torch.save(model.state_dict(), f"models/{experiment_name}_best.pt")
            print("  --> Model Saved!")
        else:
            patience_counter += 1
            if patience > 0 and patience_counter >= patience:
                print("  --> Early Stopping!")
                break

    writer.close()
    return model, history # Return both model and history

Classes found: ['HER2plus', 'Luminal_A', 'Luminal_B', 'Triple_negative']
Training patches: 2355
Validation patches: 589

Class Weights: tensor([0.9684, 0.9735, 0.6834, 2.0969], device='cuda:0')


## **🔄 Test Time Augmentation (TTA)**

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from collections import Counter
from transformers import AutoModel # Needed for PhikonModel

# --- CONFIGURATION ---
TEST_DATA_DIR = "challenge_data/test_data"  # Path to original test images
MODEL_PATH = "models/Pikton_AggressiveFT_tommalo_best.pt" # UPDATED model path to Phikon ViT
SUBMISSION_PATH = "submission_tommi15.csv"

# Patching Parameters (MUST match the training phase)
PATCH_SIZE = 224
STRIDE = 160
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- STEP 1: TTA TRANSFORMATIONS DEFINITION ---
class TTATransforms:
    """Handles Test Time Augmentation transformations."""
    def __init__(self):
        # Base normalization (required for the network)
        self.normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def get_batch(self, image_pil):
        """
        Takes a PIL image and returns a batch of 3 tensors:
        [Original, H-Flip, V-Flip]
        """
        # 1. Original
        t_orig = self.normalize(image_pil)

        # 2. Horizontal Flip
        img_h = transforms.functional.hflip(image_pil)
        t_h = self.normalize(img_h)

        # 3. Vertical Flip
        img_v = transforms.functional.vflip(image_pil)
        t_v = self.normalize(img_v)

        # Stack into a single batch (3, C, H, W)
        return torch.stack([t_orig, t_h, t_v])

# --- FLEXIBLE PREDICTOR CLASS ---
class PatchTTAPredictor:
    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.tta = TTATransforms()
        self.model.eval()

    def predict_single_slide(self, img_path, use_masks=False):
        """
        Performs inference on a slide.
        Args:
            img_path: Path to the image.
            use_masks: If True, searches for and uses the mask for filtering.
                       If False, uses the white-color heuristic.
        """
        # Load Image
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None: return None
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # Mask Handling (If requested)
        mask = None
        if use_masks:
            # Build mask filename (e.g., img_001.png -> mask_001.png)
            mask_path = img_path.parent / f"mask_{img_path.stem.replace('img_', '')}.png"
            if mask_path.exists():
                mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            else:
                # Silent fallback: if mask is missing, should we use the heuristic later?
                # Or print a warning. For now we leave mask=None.
                pass

        h, w, _ = img_rgb.shape
        if h < PATCH_SIZE or w < PATCH_SIZE:
            # If image is smaller than patch size, handle as a single patch if possible
            if h > 0 and w > 0:
                # Resize image to at least PATCH_SIZE, preserving aspect ratio
                scale = max(PATCH_SIZE / h, PATCH_SIZE / w)
                img_rgb = cv2.resize(img_rgb, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
                h, w, _ = img_rgb.shape
            else:
                return None # Cannot process empty image


        votes = []

        # Tiling Loop
        for y in range(0, h - PATCH_SIZE + 1, STRIDE):
            for x in range(0, w - PATCH_SIZE + 1, STRIDE):
                patch = img_rgb[y:y+PATCH_SIZE, x:x+PATCH_SIZE]

                is_valid_patch = False

                # --- PATCH SELECTION LOGIC ---
                if mask is not None:
                    # MASK MODE
                    patch_mask = mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
                    # 1% threshold (consistent with training)
                    tissue_ratio = np.sum(patch_mask > 0) / (PATCH_SIZE**2)
                    if tissue_ratio > 0.01:
                        is_valid_patch = True
                else:
                    # HEURISTIC MODE (or fallback if mask is missing)
                    gray = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
                    # Count white pixels
                    white_ratio = np.sum(gray > 220) / (PATCH_SIZE**2)
                    # If less than 99% is white, keep the patch
                    if white_ratio < 0.99:
                        is_valid_patch = True

                # --- INFERENCE ---
                if is_valid_patch:
                    patch_pil = Image.fromarray(patch)
                    batch_tensor = self.tta.get_batch(patch_pil).to(self.device)

                    with torch.no_grad():
                        logits = self.model(batch_tensor)
                        probs = F.softmax(logits, dim=1)

                    # TTA Soft Voting
                    avg_probs = torch.sum(probs, dim=0)
                    votes.append(torch.argmax(avg_probs).item())

        # --- MAJORITY VOTING (Final Aggregation) ---
        if len(votes) == 0:
            # Rare fallback: if no patch passes tissue filter,
            # take the center patch without filtering. This ensures a prediction is always made.
            if h >= PATCH_SIZE and w >= PATCH_SIZE:
                center_y = (h - PATCH_SIZE) // 2
                center_x = (w - PATCH_SIZE) // 2
                patch = img_rgb[center_y:center_y+PATCH_SIZE, center_x:center_x+PATCH_SIZE]
            else:
                # If image is smaller than patch size even after resize, just take the whole image (resized)
                patch = img_rgb # already resized if necessary above
                # If patch is still smaller than PATCH_SIZE, it will be padded or handled by transform
                # For now, let's just resize it directly if it's too small (e.g. 100x100)
                if patch.shape[0] < PATCH_SIZE or patch.shape[1] < PATCH_SIZE:
                    patch = cv2.resize(patch, (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_AREA)

            batch_tensor = self.tta.get_batch(Image.fromarray(patch)).to(self.device)
            with torch.no_grad():
                probs = F.softmax(self.model(batch_tensor), dim=1)
            return torch.argmax(torch.sum(probs, dim=0)).item()

        # Return the most frequent class among all patches
        return Counter(votes).most_common(1)[0][0]

# --- MODEL CLASS REDEFINITION (Necessary to load weights) ---
# This is the PhikonModel definition, copied from previous cells to ensure scope
class PhikonModel(nn.Module):
    """Phikon adapted for transfer learning."""

    def __init__(self, num_classes, dropout_rate=0.3, freeze_backbone=True):
        super().__init__()

        # Load Phikon backbone (which is a ViT-based model)
        self.backbone = AutoModel.from_pretrained("owkin/phikon")

        # Freeze backbone parameters if specified
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # Get the input features size for the classification head from the model's configuration
        # For ViT models, this is typically the hidden_size of the transformer output.
        in_features = self.backbone.config.hidden_size

        # Define a new classification head for the PhikonModel
        # This head will replace the original classification mechanism of the pretrained model
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        # Pass input through the backbone. AutoModel.from_pretrained("owkin/phikon")
        # returns a BaseModelOutputWithPooling object for ViT models.
        outputs = self.backbone(x)

        # Extract the pooled output (typically the [CLS] token's embedding after pooling),
        # which is suitable for classification. If pooler_output is None or not available,
        # fall back to the first token of the last_hidden_state.
        pooled_output = outputs.pooler_output if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None else outputs.last_hidden_state[:, 0]

        # Pass the pooled output through the custom classification head
        logits = self.classifier(pooled_output)
        return logits

    def unfreeze_last_stage(self):
        """Unfreeze the last two transformer encoder layers for fine-tuning."""
        # The ViT encoder layers are typically within `self.backbone.encoder.layer`
        # for AutoModel.from_pretrained(ViT).
        # We unfreeze the last two layers as a 'last stage' equivalent for fine-tuning.
        for param in self.backbone.encoder.layer[-2:].parameters():
            param.requires_grad = True
        print("Info: Last two transformer encoder layers unfrozen for training.")

# --- RUNNER FUNCTION ---
def run_inference_experiment(predictor, file_list, use_masks, output_csv_name, label_map):
    print(f"\n🚀 STARTING INFERENCE | Mode: {'WITH MASKS' if use_masks else 'WITHOUT MASKS (Heuristic)'}")

    predictions = []

    for img_file in tqdm(file_list):
        pred = predictor.predict_single_slide(img_file, use_masks=use_masks)
        predictions.append(pred)

    # Save CSV
    inv_map = {v: k for k, v in label_map.items()}
    pred_str = [inv_map[p] for p in predictions if p is not None] # Filter out None predictions

    # Generate sample indices only for valid predictions
    valid_files = [f for f, p in zip(file_list, predictions) if p is not None]
    sample_indices = [Path(f).name for f in valid_files]

    df = pd.DataFrame({'sample_index': sample_indices, 'label': pred_str})
    df.to_csv(output_csv_name, index=False)
    print(f"✅ Saved: {output_csv_name}")

    # --- INTEGRATED HEAD DISPLAY ---
    print(f"Preview of {output_csv_name}:")
    display(df.head())

    return df

# --- MODEL AND DATA LOADING ---

# 1. Model Setup
# Assuming DROPOUT_RATE is defined from previous cells (0.5)
# And num_classes is 4 based on the problem context.
num_classes_inference = 4 # Hardcoding for inference context if full_dataset is not available
dropout_rate_inference = 0.5 # Consistent with training

model = PhikonModel(
    num_classes=num_classes_inference,
    dropout_rate=dropout_rate_inference,
    freeze_backbone=False # Not freezing for inference model as weights are loaded anyway
)

# --- Diagnostic: List files in models directory ---
print(f"\nListing files in 'models/' directory:")
!ls -l models/

try:
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print("✅ Model loaded.")
except Exception as e:
    print(f"❌ Model Error: {e}")
    raise e
model.to(DEVICE)

# 2. Predictor and File Setup
predictor = PatchTTAPredictor(model, DEVICE)
test_path = Path(TEST_DATA_DIR)

# --- Diagnostic: List files in test directory ---
print(f"\nListing files in '{TEST_DATA_DIR}' directory:")
!ls -l "{TEST_DATA_DIR}"

# IMPORTANT FILTER: Take only images, ignoring masks in the list
all_files = list(test_path.glob("*.*")) # Corrected syntax here
test_files = sorted([f for f in all_files if f.name.startswith("img_") and f.suffix.lower() in ['.png', '.jpg', '.jpeg']])

print(f"Total files to process: {len(test_files)}")

# ⚠️ MAPPING (Always verify it matches your training!)
LABEL_MAPPING = {
    'HER2(+)': 0,
    'Luminal A': 1,
    'Luminal B': 2,
    'Triple negative': 3
}

# --- EXPERIMENT 1: WITH MASKS (Best Case) ---
df_masks = run_inference_experiment(
    predictor,
    test_files,
    use_masks=True,
    output_csv_name="submission_tommi_15.csv", # Updated submission name
    label_map=LABEL_MAPPING
)



Listing files in 'models/' directory:
total 6262079
-rw------- 1 root root 111368231 Dec 13 15:47 ConvNeXt_AggressiveFT_tommalo_best.pt
-rw------- 1 root root 111368043 Dec 10 14:22 ConvNeXt_AggressiveFT_tommi1_best.pt
-rw------- 1 root root 111368043 Dec 10 14:52 ConvNeXt_AggressiveFT_tommi2_best.pt
-rw------- 1 root root 111367855 Dec 10 12:54 ConvNeXt_AggressiveFT_tommi_best.pt
-rw------- 1 root root 111367103 Dec  9 13:56 ConvNeXt_FineTuning_fra_best.pt
-rw------- 1 root root 111367667 Dec 10 10:22 ConvNeXt_FineTuning_franci_best.pt
-rw------- 1 root root 111367855 Dec 13 15:18 ConvNeXt_FineTuning_tommalo_best.pt
-rw------- 1 root root 111367667 Dec 10 14:15 ConvNeXt_FineTuning_tommi1_best.pt
-rw------- 1 root root 111367667 Dec 10 14:45 ConvNeXt_FineTuning_tommi2_best.pt
-rw------- 1 root root 111367479 Dec 10 12:42 ConvNeXt_FineTuning_tommi_best.pt
-rw------- 1 root root 111367291 Dec  9 16:15 ConvNeXt_TL_Warmup_fra_2_best.pt
-rw------- 1 root root 111366915 Dec  9 09:43 ConvNeX

100%|██████████| 477/477 [02:09<00:00,  3.68it/s]

✅ Saved: submission_tommi_15.csv
Preview of submission_tommi_15.csv:


,sample_index,label
0,img_0000.png,Luminal A
1,img_0001.png,Luminal A
2,img_0002.png,HER2(+)
3,img_0003.png,Luminal B
4,img_0004.png,Luminal A


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from collections import Counter
from transformers import AutoModel # Needed for PhikonModel

# --- CONFIGURATION ---
TEST_DATA_DIR = "challenge_data/test_data"  # Path to original test images
MODEL_PATH = "models/Pikton_AggressiveFT1_tommalo_best.pt" # UPDATED model path to Phikon ViT
SUBMISSION_PATH = "submission_tommi17.csv"

# Patching Parameters (MUST match the training phase)
PATCH_SIZE = 224
STRIDE = 160
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- STEP 1: TTA TRANSFORMATIONS DEFINITION ---
class TTATransforms:
    """Handles Test Time Augmentation transformations."""
    def __init__(self):
        # Base normalization (required for the network)
        self.normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def get_batch(self, image_pil):
        """
        Takes a PIL image and returns a batch of 3 tensors:
        [Original, H-Flip, V-Flip]
        """
        # 1. Original
        t_orig = self.normalize(image_pil)

        # 2. Horizontal Flip
        img_h = transforms.functional.hflip(image_pil)
        t_h = self.normalize(img_h)

        # 3. Vertical Flip
        img_v = transforms.functional.vflip(image_pil)
        t_v = self.normalize(img_v)

        # Stack into a single batch (3, C, H, W)
        return torch.stack([t_orig, t_h, t_v])

# --- FLEXIBLE PREDICTOR CLASS ---
class PatchTTAPredictor:
    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.tta = TTATransforms()
        self.model.eval()

    def predict_single_slide(self, img_path, use_masks=False):
        """
        Performs inference on a slide.
        Args:
            img_path: Path to the image.
            use_masks: If True, searches for and uses the mask for filtering.
                       If False, uses the white-color heuristic.
        """
        # Load Image
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None: return None
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # Mask Handling (If requested)
        mask = None
        if use_masks:
            # Build mask filename (e.g., img_001.png -> mask_001.png)
            mask_path = img_path.parent / f"mask_{img_path.stem.replace('img_', '')}.png"
            if mask_path.exists():
                mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            else:
                # Silent fallback: if mask is missing, should we use the heuristic later?
                # Or print a warning. For now we leave mask=None.
                pass

        h, w, _ = img_rgb.shape
        if h < PATCH_SIZE or w < PATCH_SIZE:
            # If image is smaller than patch size, handle as a single patch if possible
            if h > 0 and w > 0:
                # Resize image to at least PATCH_SIZE, preserving aspect ratio
                scale = max(PATCH_SIZE / h, PATCH_SIZE / w)
                img_rgb = cv2.resize(img_rgb, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
                h, w, _ = img_rgb.shape
            else:
                return None # Cannot process empty image


        votes = []

        # Tiling Loop
        for y in range(0, h - PATCH_SIZE + 1, STRIDE):
            for x in range(0, w - PATCH_SIZE + 1, STRIDE):
                patch = img_rgb[y:y+PATCH_SIZE, x:x+PATCH_SIZE]

                is_valid_patch = False

                # --- PATCH SELECTION LOGIC ---
                if mask is not None:
                    # MASK MODE
                    patch_mask = mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
                    # 1% threshold (consistent with training)
                    tissue_ratio = np.sum(patch_mask > 0) / (PATCH_SIZE**2)
                    if tissue_ratio > 0.01:
                        is_valid_patch = True
                else:
                    # HEURISTIC MODE (or fallback if mask is missing)
                    gray = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
                    # Count white pixels
                    white_ratio = np.sum(gray > 220) / (PATCH_SIZE**2)
                    # If less than 99% is white, keep the patch
                    if white_ratio < 0.99:
                        is_valid_patch = True

                # --- INFERENCE ---
                if is_valid_patch:
                    patch_pil = Image.fromarray(patch)
                    batch_tensor = self.tta.get_batch(patch_pil).to(self.device)

                    with torch.no_grad():
                        logits = self.model(batch_tensor)
                        probs = F.softmax(logits, dim=1)

                    # TTA Soft Voting
                    avg_probs = torch.sum(probs, dim=0)
                    votes.append(torch.argmax(avg_probs).item())

        # --- MAJORITY VOTING (Final Aggregation) ---
        if len(votes) == 0:
            # Rare fallback: if no patch passes tissue filter,
            # take the center patch without filtering. This ensures a prediction is always made.
            if h >= PATCH_SIZE and w >= PATCH_SIZE:
                center_y = (h - PATCH_SIZE) // 2
                center_x = (w - PATCH_SIZE) // 2
                patch = img_rgb[center_y:center_y+PATCH_SIZE, center_x:center_x+PATCH_SIZE]
            else:
                # If image is smaller than patch size even after resize, just take the whole image (resized)
                patch = img_rgb # already resized if necessary above
                # If patch is still smaller than PATCH_SIZE, it will be padded or handled by transform
                # For now, let's just resize it directly if it's too small (e.g. 100x100)
                if patch.shape[0] < PATCH_SIZE or patch.shape[1] < PATCH_SIZE:
                    patch = cv2.resize(patch, (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_AREA)

            batch_tensor = self.tta.get_batch(Image.fromarray(patch)).to(self.device)
            with torch.no_grad():
                probs = F.softmax(self.model(batch_tensor), dim=1)
            return torch.argmax(torch.sum(probs, dim=0)).item()

        # Return the most frequent class among all patches
        return Counter(votes).most_common(1)[0][0]

# --- MODEL CLASS REDEFINITION (Necessary to load weights) ---
# This is the PhikonModel definition, copied from previous cells to ensure scope
class PhikonModel(nn.Module):
    """Phikon adapted for transfer learning."""

    def __init__(self, num_classes, dropout_rate=0.3, freeze_backbone=True):
        super().__init__()

        # Load Phikon backbone (which is a ViT-based model)
        self.backbone = AutoModel.from_pretrained("owkin/phikon")

        # Freeze backbone parameters if specified
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # Get the input features size for the classification head from the model's configuration
        # For ViT models, this is typically the hidden_size of the transformer output.
        in_features = self.backbone.config.hidden_size

        # Define a new classification head for the PhikonModel
        # This head will replace the original classification mechanism of the pretrained model
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        # Pass input through the backbone. AutoModel.from_pretrained("owkin/phikon")
        # returns a BaseModelOutputWithPooling object for ViT models.
        outputs = self.backbone(x)

        # Extract the pooled output (typically the [CLS] token's embedding after pooling),
        # which is suitable for classification. If pooler_output is None or not available,
        # fall back to the first token of the last_hidden_state.
        pooled_output = outputs.pooler_output if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None else outputs.last_hidden_state[:, 0]

        # Pass the pooled output through the custom classification head
        logits = self.classifier(pooled_output)
        return logits

    def unfreeze_last_stage(self):
        """Unfreeze the last two transformer encoder layers for fine-tuning."""
        # The ViT encoder layers are typically within `self.backbone.encoder.layer`
        # for AutoModel.from_pretrained(ViT).
        # We unfreeze the last two layers as a 'last stage' equivalent for fine-tuning.
        for param in self.backbone.encoder.layer[-2:].parameters():
            param.requires_grad = True
        print("Info: Last two transformer encoder layers unfrozen for training.")

# --- RUNNER FUNCTION ---
def run_inference_experiment(predictor, file_list, use_masks, output_csv_name, label_map):
    print(f"\n🚀 STARTING INFERENCE | Mode: {'WITH MASKS' if use_masks else 'WITHOUT MASKS (Heuristic)'}")

    predictions = []

    for img_file in tqdm(file_list):
        pred = predictor.predict_single_slide(img_file, use_masks=use_masks)
        predictions.append(pred)

    # Save CSV
    inv_map = {v: k for k, v in label_map.items()}
    pred_str = [inv_map[p] for p in predictions if p is not None] # Filter out None predictions

    # Generate sample indices only for valid predictions
    valid_files = [f for f, p in zip(file_list, predictions) if p is not None]
    sample_indices = [Path(f).name for f in valid_files]

    df = pd.DataFrame({'sample_index': sample_indices, 'label': pred_str})
    df.to_csv(output_csv_name, index=False)
    print(f"✅ Saved: {output_csv_name}")

    # --- INTEGRATED HEAD DISPLAY ---
    print(f"Preview of {output_csv_name}:")
    display(df.head())

    return df

# --- MODEL AND DATA LOADING ---

# 1. Model Setup
# Assuming DROPOUT_RATE is defined from previous cells (0.5)
# And num_classes is 4 based on the problem context.
num_classes_inference = 4 # Hardcoding for inference context if full_dataset is not available
dropout_rate_inference = 0.5 # Consistent with training

model = PhikonModel(
    num_classes=num_classes_inference,
    dropout_rate=dropout_rate_inference,
    freeze_backbone=False # Not freezing for inference model as weights are loaded anyway
)

# --- Diagnostic: List files in models directory ---
print(f"\nListing files in 'models/' directory:")
!ls -l models/

try:
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print("✅ Model loaded.")
except Exception as e:
    print(f"❌ Model Error: {e}")
    raise e
model.to(DEVICE)

# 2. Predictor and File Setup
predictor = PatchTTAPredictor(model, DEVICE)
test_path = Path(TEST_DATA_DIR)

# --- Diagnostic: List files in test directory ---
print(f"\nListing files in '{TEST_DATA_DIR}' directory:")
!ls -l "{TEST_DATA_DIR}"

# IMPORTANT FILTER: Take only images, ignoring masks in the list
all_files = list(test_path.glob("*.*")) # Corrected syntax here
test_files = sorted([f for f in all_files if f.name.startswith("img_") and f.suffix.lower() in ['.png', '.jpg', '.jpeg']])

print(f"Total files to process: {len(test_files)}")

# ⚠️ MAPPING (Always verify it matches your training!)
LABEL_MAPPING = {
    'HER2(+)': 0,
    'Luminal A': 1,
    'Luminal B': 2,
    'Triple negative': 3
}

# --- EXPERIMENT 1: WITH MASKS (Best Case) ---
df_masks = run_inference_experiment(
    predictor,
    test_files,
    use_masks=True,
    output_csv_name=SUBMISSION_PATH, # Updated submission name
    label_map=LABEL_MAPPING
)


Listing files in 'models/' directory:
total 6599636
-rw------- 1 root root 111368231 Dec 13 15:47 ConvNeXt_AggressiveFT_tommalo_best.pt
-rw------- 1 root root 111368043 Dec 10 14:22 ConvNeXt_AggressiveFT_tommi1_best.pt
-rw------- 1 root root 111368043 Dec 10 14:52 ConvNeXt_AggressiveFT_tommi2_best.pt
-rw------- 1 root root 111367855 Dec 10 12:54 ConvNeXt_AggressiveFT_tommi_best.pt
-rw------- 1 root root 111367103 Dec  9 13:56 ConvNeXt_FineTuning_fra_best.pt
-rw------- 1 root root 111367667 Dec 10 10:22 ConvNeXt_FineTuning_franci_best.pt
-rw------- 1 root root 111367855 Dec 13 15:18 ConvNeXt_FineTuning_tommalo_best.pt
-rw------- 1 root root 111367667 Dec 10 14:15 ConvNeXt_FineTuning_tommi1_best.pt
-rw------- 1 root root 111367667 Dec 10 14:45 ConvNeXt_FineTuning_tommi2_best.pt
-rw------- 1 root root 111367479 Dec 10 12:42 ConvNeXt_FineTuning_tommi_best.pt
-rw------- 1 root root 111367291 Dec  9 16:15 ConvNeXt_TL_Warmup_fra_2_best.pt
-rw------- 1 root root 111366915 Dec  9 09:43 ConvNeX

100%|██████████| 477/477 [01:48<00:00,  4.38it/s]

✅ Saved: submission_tommi_17.csv
Preview of submission_tommi_17.csv:


,sample_index,label
0,img_0000.png,Luminal A
1,img_0001.png,Luminal A
2,img_0002.png,HER2(+)
3,img_0003.png,Luminal B
4,img_0004.png,Luminal A


In [ ]:
import pandas as pd

def compare_predictions(file_path1, file_path2):
    print(f"\n--- Confronto Predizioni ---")

    df1 = None
    df2 = None

    # Load first CSV
    try:
        df1 = pd.read_csv(file_path1)
        unique_predictions1 = df1['label'].nunique()
        print(f"File: {file_path1}")
        print(f"Numero di predizioni differenti: {unique_predictions1}")
        print("Classi predette:")
        counts1 = df1['label'].value_counts().sort_index()
        for label, count in counts1.items():
            print(f"- {label}: {count} occorrenze")
    except FileNotFoundError:
        print(f"Errore: Il file '{file_path1}' non è stato trovato.")
        return

    print("\n--------------------------")

    # Load second CSV
    try:
        df2 = pd.read_csv(file_path2)
        unique_predictions2 = df2['label'].nunique()
        print(f"File: {file_path2}")
        print(f"Numero di predizioni differenti: {unique_predictions2}")
        print("Classi predette:")
        counts2 = df2['label'].value_counts().sort_index()
        for label, count in counts2.items():
            print(f"- {label}: {count} occorrenze")
    except FileNotFoundError:
        print(f"Errore: Il file '{file_path2}' non è stato trovato.")
        return

    if df1 is not None and df2 is not None:
        print("\n--- Differenze per Classe ---")
        # Combine counts and calculate differences
        all_labels = pd.Index(counts1.index.tolist() + counts2.index.tolist()).unique().sort_values()

        comparison_df = pd.DataFrame({
            'File 1 Counts': counts1.reindex(all_labels, fill_value=0),
            'File 2 Counts': counts2.reindex(all_labels, fill_value=0)
        })
        comparison_df['Difference (File 2 - File 1)'] = comparison_df['File 2 Counts'] - comparison_df['File 1 Counts']

        print(comparison_df)

# Esegui il confronto
compare_predictions("submissions/submission_WITH_MASKS_3.csv", "submissions/submission_tommi_20.csv")



--- Confronto Predizioni ---
File: submissions/submission_WITH_MASKS_3.csv
Numero di predizioni differenti: 4
Classi predette:
- HER2(+): 102 occorrenze
- Luminal A: 127 occorrenze
- Luminal B: 151 occorrenze
- Triple negative: 97 occorrenze

--------------------------
File: submissions/submission_tommi_20.csv
Numero di predizioni differenti: 4
Classi predette:
- HER2(+): 139 occorrenze
- Luminal A: 106 occorrenze
- Luminal B: 185 occorrenze
- Triple negative: 47 occorrenze

--- Differenze per Classe ---
                 File 1 Counts  File 2 Counts  Difference (File 2 - File 1)
HER2(+)                    102            139                            37
Luminal A                  127            106                           -21
Luminal B                  151            185                            34
Triple negative             97             47                           -50


In [ ]:
import pandas as pd
from collections import Counter
import os

# Define file paths for input submissions
FILE_WITH_MASKS = "submissions/submission_WITH_MASKS_3.csv"
FILE_TOMMI_16 = "submissions/submission_tommi_16.csv"
FILE_TOMMI_12_2 = "submissions/submission_tommi_12-2.csv"
OUTPUT_FILE = "submissions/submission_majority_vote.csv"

# Define the consistent label mapping (assuming it's the same across your models)
LABEL_MAPPING = {
    'HER2(+)': 0,
    'Luminal A': 1,
    'Luminal B': 2,
    'Triple negative': 3
}
INV_LABEL_MAPPING = {v: k for k, v in LABEL_MAPPING.items()}

print("Caricamento dei file di submission...")

try:
    df_masks = pd.read_csv(FILE_WITH_MASKS)
    df_tommi16 = pd.read_csv(FILE_TOMMI_16)
    df_tommi12_2 = pd.read_csv(FILE_TOMMI_12_2)
except FileNotFoundError as e:
    print(f"Errore: File non trovato - {e.filename}. Assicurati che i file di submission esistano nella directory 'submissions/'.")
    # Optionally, you could exit or raise a more specific error here
    exit()

# Ensure sample indices are consistent and merge DataFrames for easier processing
# We'll use df_masks as the base for merging, assuming all have the same sample_index
merged_df = pd.merge(df_masks, df_tommi16, on='sample_index', suffixes=('_masks', '_tommi16'))
merged_df = pd.merge(merged_df, df_tommi12_2, on='sample_index', suffixes=('', '_tommi12_2'))

# Rename the last label column for clarity
merged_df.rename(columns={'label': 'label_tommi12_2'}, inplace=True)

final_predictions_numeric = []

print("Esecuzione del majority voting...")

# Iterate through each sample (row) to apply majority voting logic
for index, row in merged_df.iterrows():
    predictions_for_sample = [
        LABEL_MAPPING[row['label_masks']],
        LABEL_MAPPING[row['label_tommi16']],
        LABEL_MAPPING[row['label_tommi12_2']]
    ]

    # Count occurrences of each prediction
    counts = Counter(predictions_for_sample)

    # Determine majority vote
    majority_vote = -1 # Initialize with an invalid label ID
    max_count = 0
    is_tie = False

    # Find the most common prediction(s)
    for pred_id, count in counts.items():
        if count > max_count:
            max_count = count
            majority_vote = pred_id
            is_tie = False
        elif count == max_count: # This means there's a tie for the current max_count
            is_tie = True

    # Apply tie-breaking rule: if all three are different (max_count=1, len(counts)=3)
    # or if there's a 2-way tie (e.g. A, B, B) and max_count is 2, the previous logic handles that.
    # The specific request is "se tutte e tre predicono diverso usa submissions/submission_WITH_MASKS_3.csv"
    if max_count == 1 and len(counts) == 3: # Case: A, B, C (all different)
        final_predictions_numeric.append(LABEL_MAPPING[row['label_masks']]) # Use prediction from FILE_WITH_MASKS
    elif is_tie and max_count == 1: # This also covers A,B,C case, but explicitly using is_tie is safer
         final_predictions_numeric.append(LABEL_MAPPING[row['label_masks']]) # Use prediction from FILE_WITH_MASKS
    else: # Clear majority (e.g., A, A, B or A, A, A)
        final_predictions_numeric.append(majority_vote)

# Convert numerical predictions back to string labels
final_labels_str = [INV_LABEL_MAPPING[p] for p in final_predictions_numeric]

# Create the final submission DataFrame
submission_df = pd.DataFrame({
    'sample_index': merged_df['sample_index'],
    'label': final_labels_str
})

# Ensure the 'submissions' directory exists
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# Save the submission file
submission_df.to_csv(OUTPUT_FILE, index=False)

print(f"✅ File di submission generato con successo: {OUTPUT_FILE}")
print(f"Anteprima di {OUTPUT_FILE}:")
display(submission_df.head())

# Optional: Print distribution of final predictions
print("\nDistribuzione delle predizioni finali:")
print(submission_df['label'].value_counts())


Caricamento dei file di submission...
Esecuzione del majority voting...
✅ File di submission generato con successo: submissions/submission_majority_vote.csv
Anteprima di submissions/submission_majority_vote.csv:


,sample_index,label
0,img_0000.png,Luminal A
1,img_0001.png,Luminal A
2,img_0002.png,Luminal A
3,img_0003.png,Luminal B
4,img_0004.png,Luminal A



Distribuzione delle predizioni finali:
label
Luminal B          164
Luminal A          123
HER2(+)            120
Triple negative     70
Name: count, dtype: int64
